In [1]:
import itertools
import mujoco
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from IPython.display import display
from scipy.spatial.transform import Rotation as R

In [2]:
def jump_full(horizontal_velocity, vertical_velocity, knee_bend, incident_angle, angular_velocity):
    xml = """
    <mujoco>
        <option gravity="0 0 -9.81"/>
        <worldbody>
            <camera name="sideview" pos="2 0 0.5" euler="0 90 90"/>
            <geom name="ground" type="plane" size="2 2 0.1" pos="0 0 0" rgba="0 0.6 0 1" friction="1 0.005 0.0001"/>
            <body name="rodd" pos="0 -0.5 1">
                <freejoint/>
                <geom name="rod" type="cylinder" pos="0 0 0" size="0.005 0.25" rgba="1 1 1 1" solref="0.0001 0.001" solimp="0.99 0.99 0.01" friction="1 0.005 0.0001"/>
            </body>
        </worldbody>
    </mujoco>
    """

    model = mujoco.MjModel.from_xml_string(xml)
    data = mujoco.MjData(model)
    renderer = mujoco.Renderer(model)

    data.qpos[0] = 0      # x
    data.qpos[1] = -0.5     # y
    data.qpos[2] = 0.01 + np.cos(incident_angle)/4      # z
    data.qpos[3] = np.cos(incident_angle / 2)  # w
    data.qpos[4] = np.sin(incident_angle / 2)  # x
    data.qpos[5] = 0                  # y
    data.qpos[6] = 0                  # z

    data.qvel[0] = 0.0   # vx
    data.qvel[1] = horizontal_velocity   # vy
    data.qvel[2] = vertical_velocity     # vz
    data.qvel[3] = angular_velocity  # wx
    data.qvel[4] = 0.0   # wy
    data.qvel[5] = 0.0   # wz

    # --- SIMULATION LOOP AND DATA COLLECTION ---
    positions = []
    linear_velocities = []
    angular_velocities = []
    external_forces = []
    saved_qpos = []
    saved_qvel = []
    num_contacts = []

    num_steps = 400
    # Get the geom ID for the rod (before the loop)
    rod_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "rod")
    for _ in range(num_steps):
        mujoco.mj_step(model, data)
        positions.append(data.xipos[1].copy())
        linear_velocities.append(data.qvel[:3].copy())
        angular_velocities.append(data.qvel[3:].copy())
        saved_qpos.append(data.qpos.copy())
        saved_qvel.append(data.qvel.copy())
        num_contacts.append(data.ncon)
        net_force = np.zeros(3)
        for i in range(data.ncon):
            contact = data.contact[i]
            if contact.geom1 == rod_geom_id or contact.geom2 == rod_geom_id:
                force = np.zeros(6)
                mujoco.mj_contactForce(model, data, i, force)
                net_force += force[:3]
        external_forces.append(net_force.copy())

    # After the simulation loop, find the first collision duration
    first_contact = None
    first_contact_end = None
    for i, n in enumerate(num_contacts):
        if n > 0 and first_contact is None:
            first_contact = i
        if first_contact is not None and n == 0:
            first_contact_end = i
            break
    if first_contact is not None and first_contact_end is not None:
        print(f"First collision lasts {first_contact_end - first_contact} steps")
    elif first_contact is not None:
        print(f"First collision starts at step {first_contact} and lasts until the end of the simulation")
    else:
        print("No collision detected in the simulation")

    # Convert to NumPy arrays
    positions = np.array(positions)
    linear_velocities = np.array(linear_velocities)
    angular_velocities = np.array(angular_velocities)
    external_forces = np.array(external_forces)
    saved_qpos = np.array(saved_qpos)
    saved_qvel = np.array(saved_qvel)


    if first_contact_end is not None:
        # Find max height after first collision ends
        z_after = positions[first_contact_end:, 2]  # z is positions[:, 0]
        if len(z_after) > 0:
            max_height_after = np.max(z_after)
            max_height_after_idx = np.argmax(z_after) + first_contact_end  # global step index
            print(f"Max height after first collision: {max_height_after:.4f} m at step {max_height_after_idx}")
        else:
            print("No steps after first collision to check for max height.")
    else:
        print("No collision end detected, cannot compute max height after collision.")

    # Incident angle
    quat_first = saved_qpos[first_contact, 3:7]  # (w, x, y, z)
    quat_first_xyzw = [quat_first[1], quat_first[2], quat_first[3], quat_first[0]]  # reorder for scipy
    rot_first = R.from_quat(quat_first_xyzw)
    rot_matrix_first = rot_first.as_matrix()
    cylinder_axis_world_first = rot_matrix_first[:, 2]  # local z-axis in world frame

    # Project onto y-z plane
    axis_yz_first = cylinder_axis_world_first[1:]  # [y, z]
    angle_rad_first = np.arctan2(axis_yz_first[1], axis_yz_first[0])  # angle from y-axis in yz plane
    angle_deg_first = np.degrees(angle_rad_first)
    print(f"Angle of cylinder axis relative to y-axis in the yz plane upon first contact (step {first_contact}): {angle_deg_first:.2f} degrees")


    # Peak angle
    quat = saved_qpos[max_height_after_idx, 3:7]  # (w, x, y, z)
    quat_xyzw = [quat[1], quat[2], quat[3], quat[0]]  # reorder for scipy
    rot = R.from_quat(quat_xyzw)
    rot_matrix = rot.as_matrix()
    cylinder_axis_world = rot_matrix[:, 2]  # local z-axis in world frame

    # Project onto y-z plane
    axis_yz = cylinder_axis_world[1:]  # [y, z]
    angle_rad = np.arctan2(axis_yz[1], axis_yz[0])  # angle from y-axis in yz plane
    angle_deg = np.degrees(angle_rad)
    print(f"Angle of cylinder axis relative to y-axis in the yz plane at step {max_height_after_idx}: {angle_deg:.2f} degrees")

    # --- ANIMATION USING SAVED STATES ---
    frames = []
    num_frames = 196  # or any number <= num_steps

    # Create a new data object for rendering (optional, for safety)
    data_render = mujoco.MjData(model)

    for i in range(num_frames):
        data_render.qpos[:] = saved_qpos[i]
        data_render.qvel[:] = saved_qvel[i]
        mujoco.mj_forward(model, data_render)  # update derived quantities
        renderer.update_scene(data_render, camera="sideview")
        frame = renderer.render()
        frames.append(frame)

    fig, ax = plt.subplots()
    im = ax.imshow(frames[0])
    ax.axis('off')

    def update(i):
        im.set_data(frames[i])
        return [im]

    ani = FuncAnimation(fig, update, frames=num_frames, interval=20, blit=True)
    plt.close(fig)
    display(HTML(ani.to_jshtml()))

In [3]:
def jump_height(horizontal_velocity, vertical_velocity, knee_bend, incident_angle, angular_velocity):
    xml = """
    <mujoco>
        <option gravity="0 0 -9.81"/>
        <worldbody>
            <camera name="sideview" pos="2 0 0.5" euler="0 90 90"/>
            <geom name="ground" type="plane" size="2 2 0.1" pos="0 0 0" rgba="0 0.6 0 1" friction="1 0.005 0.0001"/>
            <body name="rodd" pos="0 -0.5 1">
                <freejoint/>
                <geom name="rod" type="cylinder" pos="0 0 0" size="0.005 0.25" rgba="1 1 1 1" solref="0.0001 0.001" solimp="0.99 0.99 0.01" friction="1 0.005 0.0001"/>
            </body>
        </worldbody>
    </mujoco>
    """

    model = mujoco.MjModel.from_xml_string(xml)
    data = mujoco.MjData(model)
    renderer = mujoco.Renderer(model)

    data.qpos[0] = 0      # x
    data.qpos[1] = -0.5   # y
    data.qpos[2] = 0.01 + np.cos(incident_angle)/4   # z
    data.qpos[3] = np.cos(incident_angle / 2)  # w
    data.qpos[4] = np.sin(incident_angle / 2)  # x
    data.qpos[5] = 0                  # y
    data.qpos[6] = 0                  # z

    data.qvel[0] = 0.0   # vx
    data.qvel[1] = horizontal_velocity   # vy
    data.qvel[2] = vertical_velocity     # vz
    data.qvel[3] = angular_velocity  # wx
    data.qvel[4] = 0.0   # wy
    data.qvel[5] = 0.0   # wz

    # --- SIMULATION LOOP AND DATA COLLECTION ---
    positions = []
    num_contacts = []

    num_steps = 400
    # Get the geom ID for the rod (before the loop)
    rod_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "rod")
    for _ in range(num_steps):
        mujoco.mj_step(model, data)
        positions.append(data.xipos[1].copy())
        num_contacts.append(data.ncon)

    # After the simulation loop, find the first collision duration
    first_contact = None
    first_contact_end = None
    for i, n in enumerate(num_contacts):
        if n > 0 and first_contact is None:
            first_contact = i
        if first_contact is not None and n == 0:
            first_contact_end = i
            break


    # Convert to NumPy arrays
    positions = np.array(positions)


    if first_contact_end is not None:
        # Find max height after first collision ends
        z_after = positions[first_contact_end:, 2]  # z is positions[:, 0]
        if len(z_after) > 0:
            max_height_after = np.max(z_after)
            max_height_after_idx = np.argmax(z_after) + first_contact_end  # global step index
            #print(f"Max height after first collision: {max_height_after:.4f} m at step {max_height_after_idx}")
        else:
            print("No steps after first collision to check for max height.")
    else:
        print("No collision end detected, cannot compute max height after collision.")


    return max_height_after

In [4]:
horizontal_velocity_range = np.arange(5.0, 5.1, 0.5)
vertical_velocity_range = np.arange(-3.0, -2.9, 0.5)
knee_bend_range = np.arange(0.4, 0.41, 0.1)
incident_angle_range = np.deg2rad(np.arange(4, 9, 5))
angular_velocity_range = np.arange(-3, -2.9, 1)

top_jump = 0.0
jumps = 0
for hv, vv, le, ia, av in itertools.product(
        horizontal_velocity_range,
        vertical_velocity_range,
        knee_bend_range,
        incident_angle_range,
        angular_velocity_range):
    max_height_after = jump_height(hv, vv, le, ia, av)
    jumps+=1
    if max_height_after > top_jump:
        top_jump = max_height_after
        top_jump_hv = hv
        top_jump_vv = vv
        top_jump_le = le
        top_jump_ia = ia
        top_jump_av = av

print(f"Jumps: {jumps}")
print(f"Top jump height: {top_jump:.4f} m with parameters:")
print(f"    Horizontal Velocity: {top_jump_hv:.2f} m/s")
print(f"    Vertical Velocity: {top_jump_vv:.2f} m/s")
print(f"    Leg Extension: {top_jump_le:.2f}")
print(f"    Incident Angle: {np.degrees(top_jump_ia):.2f} degrees")
print(f"    Angular Velocity: {top_jump_av:.2f} rad/s")

jump_full(top_jump_hv, top_jump_vv, top_jump_le, top_jump_ia, top_jump_av)

Jumps: 1
Top jump height: 0.4045 m with parameters:
    Horizontal Velocity: 5.00 m/s
    Vertical Velocity: -3.00 m/s
    Leg Extension: 0.40
    Incident Angle: 4.00 degrees
    Angular Velocity: -3.00 rad/s
First collision lasts 7 steps
Max height after first collision: 0.4045 m at step 196
Angle of cylinder axis relative to y-axis in the yz plane upon first contact (step 2): 92.40 degrees
Angle of cylinder axis relative to y-axis in the yz plane at step 196: 163.49 degrees
